In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)



In [2]:
from datetime import timedelta

from analyse.advertising.e00_download.mediatree import (
    CachedMediatreeAPI,
    all_intervals_between,
)
from analyse.advertising.e02_finger_printer.finger_printer import (
    FingerprintPipeline,
    print_report,
    plot_groups,
)

from datetime import datetime
from zoneinfo import ZoneInfo
import json
from pathlib import Path

tz_paris = ZoneInfo("Europe/Paris")
channel = "tf1"
output_prefix = "INVERSED"

MONDAY_LAST_YEAR = datetime(2025, 3, 10, tzinfo=tz_paris)

SIMILARITY_THRESHOLD = 0.2

OUTPUT_FILE = f"{output_prefix}_report_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.json"
OUTPUT_PLOT = f"{output_prefix}_plot_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.png"


In [6]:
api = CachedMediatreeAPI()

INPUT_SEGMENTS_FOLDER = os.path.join(
    "../working_data", f"segments_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}"
)

input_files = []

CUSTOM_DATES = [
    (datetime(2025, 3, 10, 12, 0, tzinfo=tz_paris), datetime(2025, 3, 10, 12, 30, tzinfo=tz_paris)),
    (datetime(2025, 3, 11, 12, 30, tzinfo=tz_paris), datetime(2025, 3, 11, 13, 0, tzinfo=tz_paris)),
]

for start_date, end_date in CUSTOM_DATES:
    audio_file = await api.download_export(
        channel, start_date, end_date + timedelta(minutes=1)
    )  # on ajoute une minute pour être sûr de couvrir toute la période

    segment_file = os.path.join(
        INPUT_SEGMENTS_FOLDER,
        f"{start_date.strftime('%Y-%m-%d_%H-%M-%S')}.json",
    )
    input_files.append((audio_file, segment_file, start_date.timestamp()))

input_files


[('../exports/tf1_2025-03-10_11-00-00Z_2025-03-10_11-31-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-10_12-00-00.json',
  1741604400.0),
 ('../exports/tf1_2025-03-11_11-30-00Z_2025-03-11_12-01-00Z.mp3',
  '../working_data/segments_tf1_20250310/2025-03-11_12-30-00.json',
  1741692600.0)]

In [8]:
pipeline = FingerprintPipeline(
    sources=input_files,
    similarity_threshold=0.2,
)

if False:
    fingerprints = pipeline.fingerprints()
    pipeline.diagnose(fingerprints)
else:
    report, fingerprints, groups = pipeline.run()
    print_report(report)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f"  Rapport JSON : {Path(OUTPUT_FILE).absolute()}")

    # Durée totale = fin du dernier segment
    total = max(fp.end_sec for fp in fingerprints) if fingerprints else 3600
    plot_groups(report, total, OUTPUT_PLOT)
    print(f"  Graphique : {Path(OUTPUT_PLOT).absolute()}")


  [Cache] Chargement depuis tf1_2025-03-10_11-00-00Z_2025-03-10_11-31-00Z_cc104e11_86a11994.json
  [Cache] Chargement depuis tf1_2025-03-11_11-30-00Z_2025-03-11_12-01-00Z_f3d554b2_86a11994.json

[Total] 552 empreintes sur 2 source(s)

[Clustering] 552 segments — construction de l'index inversé...
  100 paires candidates → 8 retenues (≥ 2 hashes communs)  [vs 152076 paires en O(n²)]


Comparaison fingerprints: 100%|██████████| 8/8 [00:00<00:00, 25692.52it/s]

  8 paires similaires trouvées

══════════════════════════════════════════════════════════════════════
  RAPPORT DE RÉPÉTITIONS
══════════════════════════════════════════════════════════════════════
  Segments analysés : 552
  Groupes détectés  : 544

  Classification             Groupes   Occurrences tot
  ────────────────────────────────────────────────────
  SEGMENT_UNIQUE                 536               536
  CONTENU_REPETE                   8                16

  TOP 15 GROUPES LES PLUS FRÉQUENTS :
  Groupe    Occurrences   Durée moy  Classification
  ────────────────────────────────────────────────────────────
  G318                2      20.4s  CONTENU_REPETE
    ↳ [2025-03-10 11:25:14 UTC]  20.2s
    ↳ [2025-03-11 11:30:56 UTC]  20.5s
  G321                2       1.4s  CONTENU_REPETE
    ↳ [2025-03-10 11:28:07 UTC]  1.5s
    ↳ [2025-03-11 11:31:37 UTC]  1.2s
  G323                2       4.0s  CONTENU_REPETE
    ↳ [2025-03-10 11:28:16 UTC]  4.0s
    ↳ [2025-03-11 11:31:46 UT

  Visualisation sauvegardée : INVERSED_plot_tf1_20250310.png
  Graphique : /root/Workspace/quotaclimat/analyse/advertising/e02_finger_printer/INVERSED_plot_tf1_20250310.png
